# Pixelle-Video × Wan2GP on Google Colab

Runs [Pixelle-Video](https://github.com/AIDC-AI/Pixelle-Video) end-to-end — **topic → AI script → AI images/videos → TTS narration → final short video** — using the **Wan2GP in-process backend**: the image and video generation models are loaded directly in this runtime through WanGP's Python API. No ComfyUI server, no RunningHub key.

Run the cells in order. Dependencies are installed **once** (Wan2GP's `requirements.txt` + Pixelle-Video's small extras file) and shared by everything: the end-to-end generation cell, the Streamlit web UI, and plain `wgp.py`.

> **Colab VRAM note:** the free tier usually assigns a 15 GB T4 GPU. The defaults below are sized for it: **Z-Image Turbo 6B** for images and **Wan 2.1 1.3B** for video clips. On a bigger GPU (L4/A100) switch to `wan2gp/image_qwen.json`, `wan2gp/video_wan2.1_fusionx.json` or `wan2gp/video_ltx2_distilled.json`.

> **LLM note:** Pixelle-Video needs an OpenAI-compatible LLM endpoint (Qwen/DashScope, DeepSeek, OpenAI, Ollama, your own vLLM tunnel, ...) to write the narration script and image prompts. Fill it in at step 6.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime -> Change runtime type, select GPU, save, then rerun this cell.'
    ) from exc


## 2. Configure the workspace path

Choose where Wan2GP (which bundles Pixelle-Video in `Pixelle_video/`) should be installed.


In [ ]:
from pathlib import Path

WAN2GP_ROOT = Path('/content/wan2gp').resolve()
PIXELLE_ROOT = WAN2GP_ROOT / 'Pixelle_video'
print(f'Wan2GP will be installed to: {WAN2GP_ROOT}')
print(f'Pixelle-Video lives in:      {PIXELLE_ROOT}')


## 3. Download or update Wan2GP (Pixelle-Video included)

Clone the repository on the `pipeline-ltx` branch; pull the latest changes if it already exists.


In [ ]:
import subprocess

repo_url = 'https://github.com/hoangthvn2201/Wan2GP'
branch = 'pipeline-ltx'

if WAN2GP_ROOT.exists():
    print('Repository already exists. Updating...')
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', branch], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull', 'origin', branch], check=True)


## 4. Install system dependencies

Shared libraries for video and audio processing. If you see a warning about skipping an extra repository, it is safe to ignore.


In [ ]:
import os, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)


## 5. Install Python dependencies (one-time)

A **single install** covers both projects:

- `requirements.txt` (Wan2GP) — torch ecosystem, diffusers, loguru, pydantic, moviepy, ffmpeg-python, ...
- `Pixelle_video/requirements.txt` — only the Pixelle extras (streamlit, openai, edge-tts, comfykit, playwright, ...)

Afterwards Chromium is installed for Pixelle's HTML frame-template rendering. This cell takes several minutes.


In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install',
    '--force-reinstall', '--no-deps',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install', 'xformers==0.0.32.post2',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

# One-time install: Wan2GP requirements + Pixelle-Video extras in a single resolve
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '-r', str(WAN2GP_ROOT / 'requirements.txt'),
    '-r', str(PIXELLE_ROOT / 'requirements.txt')], check=True, env=env)

# Chromium for HTML frame template rendering (Pixelle composes subtitles via Playwright)
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True, env=env)

# Re-assert a modern setuptools AFTER all installs: the dependency resolve can
# remove it from /usr/local, letting Ubuntu's ancient system pkg_resources
# (which still uses pkgutil.ImpImporter, removed in Python 3.12) shadow it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'setuptools', 'wheel'], check=True, env=env)


## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend.


In [ ]:
from pathlib import Path

target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


## 6. Configure Pixelle-Video

Fill in your **LLM endpoint** (required) and adjust the media workflows / TTS voice if you like, then run the cell — it writes `Pixelle_video/config.yaml`.

The `wan2gp/...` workflows are *descriptors* that map to WanGP models (see `Pixelle_video/WAN2GP_BACKEND.md`). Model checkpoints are downloaded automatically by WanGP on first use.


In [ ]:
import yaml

# --- LLM (required: writes the narration script & image prompts) -----------
LLM_API_KEY  = ''                                  # <-- your API key
LLM_BASE_URL = 'https://api.deepseek.com'          # any OpenAI-compatible endpoint
LLM_MODEL    = 'deepseek-chat'

# --- Media generation (wan2gp = models loaded in-process by WanGP) ---------
IMAGE_WORKFLOW = 'wan2gp/image_z_image.json'       # Z-Image Turbo 6B  (T4-friendly)
VIDEO_WORKFLOW = 'wan2gp/video_wan2.1_1.3B.json'   # Wan 2.1 1.3B t2v  (T4-friendly)
# Bigger GPUs:
#   IMAGE_WORKFLOW = 'wan2gp/image_qwen.json'             # Qwen Image 20B
#   VIDEO_WORKFLOW = 'wan2gp/video_wan2.1_fusionx.json'   # Wan 2.1 FusioniX 14B
#   VIDEO_WORKFLOW = 'wan2gp/video_ltx2_distilled.json'   # LTX-2 22B (video + audio)

PROMPT_PREFIX = ('Minimalist black-and-white matchstick figure style illustration, '
                 'clean lines, simple sketch style')

# --- TTS (local edge-tts, runs on CPU) --------------------------------------
TTS_VOICE = 'en-US-GuyNeural'    # e.g. 'zh-CN-YunjianNeural' for Chinese
TTS_SPEED = 1.2

# --- WanGP runtime -----------------------------------------------------------
WAN2GP_CLI_ARGS = ['--profile', '5']   # low-VRAM profile (same as the official notebook)

config = {
    'project_name': 'Pixelle-Video',
    'llm': {
        'api_key': LLM_API_KEY,
        'base_url': LLM_BASE_URL,
        'model': LLM_MODEL,
        'enable_thinking': False,
    },
    'comfyui': {
        'comfyui_url': 'http://127.0.0.1:8188',
        'comfyui_api_key': None,
        'runninghub_api_key': None,
        'runninghub_concurrent_limit': 1,
        'runninghub_instance_type': None,
        'tts': {
            'inference_mode': 'local',
            'local': {'voice': TTS_VOICE, 'speed': TTS_SPEED},
            'comfyui': {'default_workflow': None},
        },
        'image': {'default_workflow': IMAGE_WORKFLOW, 'prompt_prefix': PROMPT_PREFIX},
        'video': {'default_workflow': VIDEO_WORKFLOW, 'prompt_prefix': PROMPT_PREFIX},
    },
    'wan2gp': {
        'root': str(WAN2GP_ROOT),
        'cli_args': WAN2GP_CLI_ARGS,
        'output_dir': None,
    },
    'template': {'default_template': '1080x1920/image_default.html'},
}

(PIXELLE_ROOT / 'config.yaml').write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
print('config.yaml written:')
print((PIXELLE_ROOT / 'config.yaml').read_text())


## 7. Initialize Pixelle-Video

Sets up paths and initializes the core services. The WanGP session itself is created lazily — model weights are only loaded (and downloaded) on the first generation.


In [ ]:
import os, sys

os.chdir(PIXELLE_ROOT)                                   # relative paths: workflows/, templates/, output/
os.environ['PIXELLE_VIDEO_ROOT'] = str(PIXELLE_ROOT)

for p in (str(WAN2GP_ROOT), str(PIXELLE_ROOT)):          # Pixelle_video.* + shared.api imports
    if p not in sys.path:
        sys.path.insert(0, p)

from pixelle_video.service import pixelle_video

await pixelle_video.initialize()
print(pixelle_video)
print('wan2gp media workflows:', [w for w in pixelle_video.media.available if w.startswith('wan2gp/')])


## 8. Generate a video end-to-end

Enter a topic and run. The pipeline will: write the script (LLM) → synthesize narration (edge-tts) → generate one image/video per scene (**WanGP, in-process**) → compose frames with the HTML template → concatenate into `final.mp4`.

> The **first** generation downloads the model checkpoint (a few GB) — subsequent scenes reuse the model already loaded in VRAM.
>
> Template naming controls the media type: `image_*.html` templates use the image model, `video_*.html` templates use the video model, `static_*.html` use no media model at all.


In [ ]:
TOPIC    = 'Why should you build a daily reading habit?'
N_SCENES = 3
TEMPLATE = '1080x1920/image_default.html'   # try '1080x1920/video_default.html' for AI video scenes

from pixelle_video.utils.template_util import resolve_template_path, get_template_type
from pixelle_video.services.frame_html import HTMLFrameGenerator

template_type = get_template_type(TEMPLATE.split('/')[-1])    # 'static' | 'image' | 'video'
media_workflow = {'image': IMAGE_WORKFLOW, 'video': VIDEO_WORKFLOW}.get(template_type)
media_width, media_height = HTMLFrameGenerator(resolve_template_path(TEMPLATE)).get_media_size()
print(f'Template type: {template_type} | media workflow: {media_workflow} | media size: {media_width}x{media_height}')

def on_progress(event):
    frame = f' frame {event.frame_current}/{event.frame_total}' if event.frame_current else ''
    action = f' [{event.action}]' if getattr(event, 'action', None) else ''
    print(f'[{event.progress * 100:5.1f}%] {event.event_type}{frame}{action}')

result = await pixelle_video.generate_video(
    text=TOPIC,
    mode='generate',
    n_scenes=N_SCENES,
    frame_template=TEMPLATE,
    media_workflow=media_workflow,
    media_width=media_width,
    media_height=media_height,
    prompt_prefix=PROMPT_PREFIX,
    tts_inference_mode='local',
    tts_voice=TTS_VOICE,
    tts_speed=TTS_SPEED,
    progress_callback=on_progress,
)

print()
print(f'Final video: {result.video_path}')
print(f'Duration:    {result.duration:.1f}s | Size: {result.file_size / 1e6:.1f} MB')


### Preview the result


In [ ]:
from IPython.display import Video

Video(result.video_path, embed=True, width=320)


## 9. (Optional) Launch the Pixelle-Video Web UI

Starts the Streamlit interface and exposes it through a free Cloudflare quick tunnel (Colab cannot expose ports directly). Click the printed `trycloudflare.com` link; keep the cell running while you use the UI and press **Stop** when done.

The web UI shares the same process-wide setup, so `wan2gp/...` workflows appear in the image/video dropdowns there too.

> **If the page errors with `Failed to fetch dynamically imported module`:** do a **hard refresh** (`Ctrl+Shift+R` / `Cmd+Shift+R`) — the browser cached the UI of a previous run whose JS chunks no longer exist. If it persists, stop and re-run this cell to get a fresh tunnel (the free tunnel occasionally drops chunk requests; the cell already forces the more reliable http2 transport).


In [ ]:
import os, re, subprocess, sys

# Cloudflare quick tunnel binary
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/usr/local/bin/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

env = os.environ.copy()
env['PIXELLE_VIDEO_ROOT'] = str(PIXELLE_ROOT)
env['PYTHONPATH'] = f"{WAN2GP_ROOT}:{PIXELLE_ROOT}"

streamlit_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'web/app.py',
     '--server.port', '8501', '--server.headless', 'true',
     '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
     '--browser.gatherUsageStats', 'false'],
    cwd=str(PIXELLE_ROOT), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501',
     '--protocol', 'http2', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print('Waiting for the tunnel URL...')
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        print(f'\n🌐 Pixelle-Video Web UI: {match.group(0)}\n')
        break

try:
    for line in iter(streamlit_proc.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping...')
finally:
    streamlit_proc.terminate()
    tunnel_proc.terminate()
    print('Web UI and tunnel stopped.')


## Notes & troubleshooting

- **Outputs** land in `Pixelle_video/output/<task_id>/final.mp4` (per-frame assets in `frames/`).
- **First generation is slow**: WanGP downloads the model checkpoint, then keeps it loaded in VRAM — later scenes are much faster.
- **Switching models** (e.g. an `image_*` run followed by a `video_*` run) makes WanGP unload/reload models — expected.
- **Out of VRAM / RAM on T4**: stick to `image_z_image` + `video_wan2.1_1.3B`, keep `--profile 5`, and use smaller media sizes (the template's media size is capped automatically by each descriptor's `max_pixels`).
- **Reasoning LLMs** (MiniMax-M3, DeepSeek-R1, Qwen3, ...): hidden chain-of-thought counts against the token budget, which can yield `LLM returned no content` on short completions. Pixelle now retries once with a larger budget automatically. Note `enable_thinking: false` is only honored by vLLM/SGLang-style servers - Ollama endpoints ignore it.
- **`AttributeError: module 'pkgutil' has no attribute 'ImpImporter'`** when launching the UI: an old system `pkg_resources` is shadowing the modern one on Python 3.12. Run `pip install --upgrade setuptools wheel` and restart (the install cell now does this automatically).
- **Add your own model**: copy a descriptor in `Pixelle_video/workflows/wan2gp/` and set `model_type` to any file name from `defaults/*.json` (e.g. `flux`, `t2v_2_2`, `ltx2_22B_distilled`).
- Full backend documentation: `Pixelle_video/WAN2GP_BACKEND.md`.
